# Lesson 07: Persistent vector search with sqlitesearch

Part A ingests (embed everything once, write `faq_vectors2.db`). Part B reopens the file without re-embedding and runs RAG.

## Part A: ingestion (run once)

In [1]:
from embedder import Embedder
from ingest import load_faq_data
from tqdm.auto import tqdm
import numpy as np, time

model = Embedder()
documents = load_faq_data()
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

batch_size = 50
vectors = []
for i in tqdm(range(0, len(texts), batch_size)):
    vectors.extend(model.encode_batch(texts[i:i + batch_size]))
vectors = np.array(vectors)
vectors.shape

C:\Users\vasus\Projects\agent-ai-bootcamp\module2-vector-search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0/30 [00:00<?, ?it/s]

  3%|▎         | 1/30 [00:01<00:32,  1.12s/it]

  7%|▋         | 2/30 [00:02<00:34,  1.22s/it]

 10%|█         | 3/30 [00:03<00:31,  1.17s/it]

...

 93%|█████████▎| 28/30 [00:50<00:04,  2.47s/it]

 97%|█████████▋| 29/30 [00:53<00:02,  2.48s/it]

100%|██████████| 30/30 [00:53<00:00,  1.86s/it]

100%|██████████| 30/30 [00:53<00:00,  1.79s/it]

(1456, 384)

In [2]:
import os
from sqlitesearch import VectorSearchIndex

for f in ["faq_vectors2.db"]:
    if os.path.exists(f):
        os.remove(f)   # fresh index for a clean re-run

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

t0 = time.time()
vs_index.fit(vectors, documents)
print(f"fit: {time.time() - t0:.1f} s, file size: {os.path.getsize('faq_vectors2.db') / 1e6:.1f} MB")

fit: 0.7 s, file size: 5.6 MB


In [3]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)
[(d["course"], d["question"]) for d in results]

[('llm-zoomcamp', 'I just discovered the course. Can I still join?'),
 ('machine-learning-zoomcamp',
  'The course has already started. Can I still join it?'),
 ('data-engineering-zoomcamp',
  'Course: Can I get support if I take the course in the self-paced mode?'),
 ('ai-dev-tools-zoomcamp', 'Am I too late to join the course?'),
 ('data-engineering-zoomcamp',
  'Course - Can I follow the course after it finishes?')]

In [4]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
[d["question"] for d in results]

['I just discovered the course. Can I still join?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'Can I run the course locally instead of Codespaces?',
 'Does the course certificate show the number of course hours?',
 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?']

In [5]:
vs_index.close()

## Part B: reopen without re-embedding
(In the course this is a new notebook / new Python session. `model` here only encodes queries.)

In [6]:
from embedder import Embedder
from sqlitesearch import VectorSearchIndex

model = Embedder()

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)
[(d["course"], d["question"]) for d in results]

[('data-engineering-zoomcamp',
  'Java Kafka: How to run producer/consumer/kstreams/etc in terminal'),
 ('data-engineering-zoomcamp',
  'Module “kafka” not found when trying to run producer.py'),
 ('data-engineering-zoomcamp',
  'How to fix the error "ModuleNotFoundError: No module named \'kafka.vendor.six.moves\'?"'),
 ('data-engineering-zoomcamp',
  'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent'),
 ('data-engineering-zoomcamp',
  'kafka.errors.NoBrokersAvailable: NoBrokersAvailable')]

## RAG on the persistent vector index

In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")   # keys live in the project root .env

# Course: openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [8]:
from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [9]:
print(vector_assistant.rag("the program has already begun, can I still sign up?"))

Yes—you can still join even though the program has already started. You’re welcome to sign up and begin working on the material right away. If you’d like to receive a certificate, just make sure you submit your project before the submission window closes (i.e., while we’re still accepting project submissions).


In [10]:
vs_index.close()